In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar `creditsight` sin instalarlo: el código vive en ejemplo/src
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 08 · Registro del modelo

En la etapa 6 hubo **muchas ejecuciones** (una por ensayo): eso es **tracking**. Ahora hay **un modelo final**, y lo que se
guarda es otra cosa: una **versión del modelo**, con su firma, su ejemplo de entrada, sus métricas de la prueba y un
**alias** (`champion`) que dice cuál es el vigente.

| | Tracking | Registro |
|---|---|---|
| Qué guarda | cada ensayo: parámetros y métricas | el modelo final, versionado |
| Cuántos hay | muchos | pocos |
| Para qué | comparar y reproducir | saber **cuál** es el modelo en uso y poder volver a una versión anterior |

> **Seguridad:** un *pickle* **puede ejecutar código al cargarse**. Solo se cargan modelos que uno mismo registró.

## 0. Conexión con el seguimiento

Dentro del servidor de Databricks no hay que configurar nada. Desde el computador se lee el `.env` (`PROFILE`). Si no hay
conexión, o se agotó la cuota, se usa MLflow **local** (`MLFLOW_FORCE_LOCAL=1`): el registro local cuenta igual y el notebook
funciona igual. Esa es la mitad del plan B.

In [2]:
import json
import warnings
from datetime import datetime

import joblib
import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature

from creditsight.config import GROUP_COL, MODELS_DIR, REPORTS_DIR, TARGET
from creditsight.data.cleaning import clean, load_raw
from creditsight.data.split import split_by_customer
from creditsight.inference.predict import required_columns
from creditsight.tracking.mlflow_utils import configure_mlflow, registered_model_name, set_experiment

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    donde = configure_mlflow()
for w in caught:
    print("Aviso:", w.message)
experimento = set_experiment("creditsight-registro")
NOMBRE = registered_model_name("creditsight_credit_score")
print("MLflow:", donde)
print("Experimento:", experimento)
print("Nombre del modelo en el registro:", NOMBRE)

2026/10/29 14:09:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/29 14:09:38 INFO mlflow.store.db.utils: Updating database tables


2026/10/29 14:09:41 INFO mlflow.tracking.fluent: Experiment with name 'creditsight-registro' does not exist. Creating a new experiment.


MLflow: local
Experimento: creditsight-registro
Nombre del modelo en el registro: creditsight_credit_score


## 1. Cargar el modelo final y los resultados de la prueba

El pipeline final lo dejó el notebook 07 (`models/final_pipeline.joblib`, **no se versiona**: lo que viaja con el repo es el
`model.joblib` que se exporta en el notebook 09 desde el registro). Las métricas salen de `07_resultados.json`.

In [3]:
modelo_final = joblib.load(MODELS_DIR / "final_pipeline.joblib")
resultados = json.loads((REPORTS_DIR / "07_resultados.json").read_text(encoding="utf-8"))
decisiones = json.loads((REPORTS_DIR / "07_decisiones_congeladas.json").read_text(encoding="utf-8"))
print("modelo:", resultados["modelo_final"], "· calibrado:", decisiones["calibrar"])
print("decisiones congeladas el:", decisiones["fecha_y_hora"])

train, _ = split_by_customer(clean(load_raw()), test_size=0.2)
numericas, categoricas = required_columns()
entrada = train[numericas + categoricas].head(200).copy()
entrada[numericas] = entrada[numericas].astype("float64")      # un entero con vacíos no cabe en la firma: todo numérico a float
print("firma sobre", len(entrada), "filas de ejemplo,", entrada.shape[1], "columnas de entrada")

modelo: random forest · calibrado: False
decisiones congeladas el: 2026-10-28T17:25:46


firma sobre 200 filas de ejemplo, 21 columnas de entrada


## 2. Qué se registra

Junto al modelo viajan:
- los **parámetros** y las **decisiones** (qué modelo, si se calibró, con qué datos se decidió);
- las **métricas de la prueba**, la misma tabla del notebook 07;
- la **firma** (columnas y tipos de entrada, probabilidades de salida) y un **ejemplo de entrada**;
- etiquetas para encontrarlo después.

> **Nota técnica:** hay que pasar `serialization_format="cloudpickle"`. El formato por defecto de MLflow rechaza guardar un
> `Pipeline` con transformadores propios (error *untrusted types*).

In [4]:
parametros = decisiones["parametros"] if isinstance(decisiones["parametros"], dict) else {"configuracion": "de partida"}
metricas_prueba = {f"test_{k}": float(v) for k, v in resultados["metricas_modelo"].items()}
firma = infer_signature(entrada, modelo_final.predict_proba(entrada))

with mlflow.start_run(run_name="modelo-final") as corrida:
    mlflow.log_params({"modelo": resultados["modelo_final"], "calibrado": decisiones["calibrar"],
                       **{f"param_{k}": v for k, v in parametros.items()}})
    mlflow.log_metrics(metricas_prueba)
    mlflow.set_tags({"etapa": "08_registro", "decisiones_congeladas": decisiones["fecha_y_hora"],
                     "unidad_de_particion": GROUP_COL, "supera_baseline": str(resultados["supera_al_baseline_con_confianza"])})
    info = mlflow.sklearn.log_model(
        modelo_final, name="model", signature=firma, input_example=entrada.head(5),
        serialization_format="cloudpickle", registered_model_name=NOMBRE,
    )
    id_corrida = corrida.info.run_id

cliente = MlflowClient()
version = max(int(v.version) for v in cliente.search_model_versions(f"name='{NOMBRE}'"))
print("ejecución:", id_corrida)
print(f"modelo registrado: {NOMBRE} · versión {version}")

2026/10/29 14:09:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/10/29 14:10:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Successfully registered model 'creditsight_credit_score'.
Created version '1' of model 'creditsight_credit_score'.


ejecución: a52daa3a33084629bda724f06db98cd7
modelo registrado: creditsight_credit_score · versión 1


## 3. Poner el alias `champion`

Una **versión** es un número que no dice nada sobre su uso. El **alias** sí: `champion` es "el modelo vigente". Quien
consuma el modelo pide `models:/<nombre>@champion` y no necesita saber qué número de versión es. Si mañana hay una versión
mejor, se mueve el alias y nadie cambia su código; si algo sale mal, se mueve de vuelta.

In [5]:
cliente.set_registered_model_alias(NOMBRE, "champion", str(version))
cliente.set_model_version_tag(NOMBRE, str(version), "decisiones_congeladas", decisiones["fecha_y_hora"])
champion = cliente.get_model_version_by_alias(NOMBRE, "champion")
print(f"alias champion → versión {champion.version}  (ejecución {champion.run_id[:8]}…)")

alias champion → versión 1  (ejecución a52daa3a…)


## 4. Comprobación de ida y vuelta

Se **carga desde el registro** (por alias) y se comprueba que predice **exactamente igual** que el objeto que se registró.
Si no, lo que está en el registro no es lo que se evaluó.

In [6]:
desde_registro = mlflow.sklearn.load_model(f"models:/{NOMBRE}@champion")
esperado = modelo_final.predict_proba(train[numericas + categoricas].head(500))
obtenido = desde_registro.predict_proba(train[numericas + categoricas].head(500))
ida_y_vuelta = bool(np.allclose(esperado, obtenido))
print("las 500 predicciones coinciden:", ida_y_vuelta, f"(diferencia máxima {np.abs(esperado - obtenido).max():.2e})")
assert ida_y_vuelta, "El modelo del registro no coincide con el que se evaluó"

las 500 predicciones coinciden: True (diferencia máxima 2.22e-16)


## 5. Guardar la constancia

Queda en `reports/08_registro.json` (y se resume en `reports/08_registro.md`): dónde se registró, qué versión, qué alias y
con qué decisiones. La app **no consulta el registro**; trabaja con el `model.joblib` que se exporta en el notebook 09.

In [7]:
constancia = {
    "fecha_y_hora": datetime.now().isoformat(timespec="seconds"), "donde": donde, "experimento": experimento,
    "modelo_registrado": NOMBRE, "version": version, "alias": "champion", "ejecucion": id_corrida,
    "ida_y_vuelta_ok": ida_y_vuelta, "modelo": resultados["modelo_final"], "calibrado": decisiones["calibrar"],
    "decisiones_congeladas": decisiones["fecha_y_hora"],
}
(REPORTS_DIR / "08_registro.json").write_text(json.dumps(constancia, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(constancia, indent=2, ensure_ascii=False))

{
  "fecha_y_hora": "2026-10-29T14:10:06",
  "donde": "local",
  "experimento": "creditsight-registro",
  "modelo_registrado": "creditsight_credit_score",
  "version": 1,
  "alias": "champion",
  "ejecucion": "a52daa3a33084629bda724f06db98cd7",
  "ida_y_vuelta_ok": true,
  "modelo": "random forest",
  "calibrado": false,
  "decisiones_congeladas": "2026-10-28T17:25:46"
}


## Qué sigue

**Notebook 09:** exportar `models/model.joblib` desde `champion`, escribir el esquema de entrada y comprobar que la función de
predicción rechaza los datos imposibles. Con eso la app ya puede cargar el modelo.